In [1]:
import embedding
from transformers import BertTokenizer
from tqdm import tqdm
import torch

c:\Users\SOHAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
import pickle
import torch
with open(r"C:\Users\SOHAM\Desktop\Entity_Aspect_Linking\EntityAspectLinking\Experiment_trainsmall\Sentence\picklefiles\eal_trainsmall.pkl", 'rb') as eal:
    data = pickle.load(eal)

ent = [data[i][0] for i in range(len(data))]

asp = [data[i][1] for i in range(len(data))]

In [11]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
target_emb = torch.zeros(len(ent), 100)
target_emb.to(device)
pretrained = 'bert-base-uncased'
ent_emb = embedding.EntityEmbedding(pretrained = pretrained, device = device)
for i in range(len(ent)):
    entity_word = ent[i]['target_entity']
    tokens = tokenizer.tokenize(entity_word)
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_ids = torch.tensor(input_ids).unsqueeze(0).to(device)
    target_emb[i] = ent_emb(input_ids)

In [12]:
with open(r"C:\Users\SOHAM\Desktop\Entity_Aspect_Linking\EntityAspectLinking\Experiment_trainsmall\Sentence\picklefiles\targetentemb_trainsmall.pkl", 'wb') as f:
    pickle.dump(target_emb, f)
print("Dumped")
f.close()

Dumped


In [13]:
import re
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]","",text)
    text = re.sub(r"(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:'\".,<>?«»“”‘’]))","",text)
    text = re.sub("<(\"[^\"]*\"|'[^']*'|[^'\">])*>","",text)    
    return text

In [14]:
taspc = 0
for el in asp:
    taspc += len(el['candidate_aspects'])

In [15]:
i = 0
j = 0
check = 0
yo = 0
unid_ls = []
while(i < taspc):
    aspect = asp[j]['true_aspect']
    i+=1    
    this = 0
    for cand in asp[j]['candidate_aspects']:
        if i >= taspc:
            break
        if cand['aspect_name'] == aspect:
            check += 1
        if cand['aspect_name'] != aspect:
            this += 1
            casp = cand['aspect_name']
            i+=1
    if this == len(asp[j]['candidate_aspects']):
        yo += 1  
        unid_ls.append(j)     
            
    j += 1
unid_ls

[3538, 3812, 4460, 4461]

In [16]:
i = 0
j = 0
check = 0
yo = 0
while(i < taspc):
    aspect = asp[j]['true_aspect']
    tokens = tokenizer.tokenize(aspect)
    i+=1    
    this = 0
    for cand in asp[j]['candidate_aspects']:
        if i >= taspc:
            break
        if j in unid_ls and aspect in str(cand["aspect_name"]): 
            i += 1          
            continue
        if len(cand['aspect_name']) == 0:
            i += 1
            continue     
        if cand['aspect_name'] == aspect:
            check += 1
        if cand['aspect_name'] != aspect:
            this += 1
            casp = cand['aspect_name']
            tokens = tokenizer.tokenize(casp)
            i+=1
    if this == len(asp[j]['candidate_aspects']):
        yo += 1           
        print("Found it", aspect, j)
            
    j += 1


In [17]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
asp_emb = torch.zeros(taspc, 100, dtype=torch.float).to(device)
i = 0
j = 0
check = 0
pbar = tqdm(total=taspc, desc="Processing")
while(i < taspc):
    aspect = asp[j]['true_aspect']
    tokens = tokenizer.tokenize(aspect)
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_ids = torch.tensor(input_ids).unsqueeze(0).to(device)
    asp_emb[i] = ent_emb(input_ids, dim = 100)
    i += 1
    pbar.update(1)
    for cand in asp[j]['candidate_aspects']:
        if i >= taspc:
            break
        if j in unid_ls and aspect in str(cand["aspect_name"]):
            i += 1
            continue
        if len(cand['aspect_name']) == 0:
            i += 1
            continue         
        if cand['aspect_name'] != aspect:
            casp = cand['aspect_name']
            tokens = tokenizer.tokenize(casp)
            input_ids = tokenizer.convert_tokens_to_ids(tokens)
            input_ids = torch.tensor(input_ids).unsqueeze(0).to(device)
            asp_emb[i] = ent_emb(input_ids, dim = 100)
            i+=1
            pbar.update(1)
            check += 1      
    j += 1
        

Processing: 100%|█████████▉| 36195/36204 [41:40<00:00, 14.57it/s] 

In [19]:
with open(r"C:\Users\SOHAM\Desktop\Entity_Aspect_Linking\EntityAspectLinking\Experiment_trainsmall\Sentence\picklefiles\aspemb_trainsmall.pkl", 'wb') as f:
    pickle.dump(asp_emb, f)
print("Dumped")
f.close()

Dumped
